# Amazon Fraud Dataset Investigation 

It loads the actual `FraudAmazon.zip / Amazon.mat` dataset and investigates the dataset only.  
Special attention is given to Amazon-specific problems that can silently produce incorrect statistics.


In [1]:
# CELL 1 — LOAD ACTUAL AMAZON DATASET
import os, zipfile, urllib.request, tempfile
import numpy as np
import pandas as pd
import scipy.io as sio
import scipy.sparse as sp

URL = "https://data.dgl.ai/dataset/FraudAmazon.zip"
# Prefer the user-supplied file; otherwise download the official DGL archive.
candidates = ["/mnt/data/FraudAmazon.zip", "/kaggle/working/FraudAmazon.zip", "FraudAmazon.zip"]
zip_path = next((p for p in candidates if os.path.exists(p)), None)
if zip_path is None:
    zip_path = os.path.join(tempfile.gettempdir(), "FraudAmazon.zip")
    urllib.request.urlretrieve(URL, zip_path)

extract_dir = os.path.join(tempfile.gettempdir(), "FraudAmazon_extracted")
os.makedirs(extract_dir, exist_ok=True)
with zipfile.ZipFile(zip_path) as z:
    z.extractall(extract_dir)
mat_path = os.path.join(extract_dir, "Amazon.mat")
amz = sio.loadmat(mat_path)

labels = np.asarray(amz["label"]).reshape(-1).astype(np.int32)
features = amz["features"]
net_upu = amz["net_upu"]
net_usu = amz["net_usu"]
net_uvu = amz["net_uvu"]

print("Loaded:", mat_path)
print("Keys:", [k for k in amz.keys() if not k.startswith("__")])

Loaded: /tmp/FraudAmazon_extracted/Amazon.mat
Keys: ['homo', 'net_upu', 'net_usu', 'net_uvu', 'features', 'label']


In [2]:
# CELL 2 — VERIFY RAW DATA
print("===== AMAZON RAW DATA CHECK =====")
print("Labels shape:", labels.shape)
print("Features shape:", features.shape)
print("UPU shape / nnz:", net_upu.shape, net_upu.nnz)
print("USU shape / nnz:", net_usu.shape, net_usu.nnz)
print("UVU shape / nnz:", net_uvu.shape, net_uvu.nnz)
print("Raw label-array counts:", dict(zip(*np.unique(labels, return_counts=True))))
print("IMPORTANT: CARE-GNN treats node IDs 0–3304 as UNLABELLED; evaluation uses 3305 onward.")

===== AMAZON RAW DATA CHECK =====
Labels shape: (11944,)
Features shape: (11944, 25)
UPU shape / nnz: (11944, 11944) 351216
USU shape / nnz: (11944, 11944) 7132958
UVU shape / nnz: (11944, 11944) 2073474
Raw label-array counts: {np.int32(0): np.int64(11123), np.int32(1): np.int64(821)}
IMPORTANT: CARE-GNN treats node IDs 0–3304 as UNLABELLED; evaluation uses 3305 onward.


In [3]:
# CELL 3 — NUMERICAL CHARACTERISTICS
num_nodes = labels.size
num_features = features.shape[1]
unlabelled_count = 3305
labelled_mask = np.arange(num_nodes) >= unlabelled_count
labelled_nodes = int(labelled_mask.sum())
fraud_count = int((labels[labelled_mask] == 1).sum())
normal_count = int((labels[labelled_mask] == 0).sum())
fraud_pct = 100 * fraud_count / labelled_nodes
imbalance_ratio = normal_count / fraud_count

rels = [net_upu.astype(np.int8), net_usu.astype(np.int8), net_uvu.astype(np.int8)]
union = rels[0] + rels[1] + rels[2]
union.data[:] = 1
union.setdiag(0); union.eliminate_zeros()
unique_undirected_edges = sp.triu(union, k=1).nnz

print("===== AMAZON STEP 2 =====")
print("Nodes (all graph nodes):", num_nodes)
print("Labelled nodes used by CARE-GNN:", labelled_nodes)
print("Unlabelled nodes:", unlabelled_count)
print("Features:", num_features)
print("Fraud/anomaly nodes (labelled subset):", fraud_count)
print("Normal nodes (labelled subset):", normal_count)
print(f"Fraud % among labelled nodes: {fraud_pct:.4f}%")
print(f"Imbalance ratio (normal:fraud): {imbalance_ratio:.4f}:1")
print("Raw relation entries:", sum(a.nnz for a in rels))
print("Union stored entries:", union.nnz)
print("Unique undirected union edges:", unique_undirected_edges)

===== AMAZON STEP 2 =====
Nodes (all graph nodes): 11944
Labelled nodes used by CARE-GNN: 8639
Unlabelled nodes: 3305
Features: 25
Fraud/anomaly nodes (labelled subset): 821
Normal nodes (labelled subset): 7818
Fraud % among labelled nodes: 9.5034%
Imbalance ratio (normal:fraud): 9.5225:1
Raw relation entries: 9557648
Union stored entries: 8835152
Unique undirected union edges: 4417576


In [4]:
# CELL 4 — GRAPH STRUCTURE AND ANOMALY ORIGIN
print("===== AMAZON GRAPH STRUCTURE =====")
print("Graph type: Homogeneous node type, multi-relational")
print("Node types: 1 (User)")
print("Relation/edge types: 3")
print("UPU: users reviewing at least one same product")
print("USU: users having at least one same star rating within one week")
print("UVU: users with top-5% mutual review text similarity")
print("Static / Dynamic: Static (no temporal graph evolution encoded)")
print("Anomaly origin: Real-world / non-injected proxy labels from Amazon review data; not synthetically injected anomalies.")

===== AMAZON GRAPH STRUCTURE =====
Graph type: Homogeneous node type, multi-relational
Node types: 1 (User)
Relation/edge types: 3
UPU: users reviewing at least one same product
USU: users having at least one same star rating within one week
UVU: users with top-5% mutual review text similarity
Static / Dynamic: Static (no temporal graph evolution encoded)
Anomaly origin: Real-world / non-injected proxy labels from Amazon review data; not synthetically injected anomalies.


In [5]:
# CELL 5 — GLOBAL HETEROPHILY: different-label edges / total labelled edges
def global_h(adj, name):
    coo = adj.tocoo(copy=False)
    m = (coo.row < coo.col) & labelled_mask[coo.row] & labelled_mask[coo.col]
    s, t = coo.row[m], coo.col[m]
    diff = int(np.count_nonzero(labels[s] != labels[t]))
    total = len(s)
    return {"Relation":name,"Total labelled edges":total,"Different-label edges":diff,
            "Same-label edges":total-diff,"Global heterophily":diff/total if total else np.nan}

global_results = pd.DataFrame([
    global_h(net_upu,"U-P-U"), global_h(net_usu,"U-S-U"),
    global_h(net_uvu,"U-V-U"), global_h(union,"Combined union")])
print("===== GLOBAL HETEROPHILY =====")
print("Protocol: undirected; reverse pairs counted once; self-loops excluded; BOTH endpoints must be labelled.")
print(global_results.to_string(index=False, formatters={"Global heterophily":lambda x:f"{x:.6f}"}))

===== GLOBAL HETEROPHILY =====
Protocol: undirected; reverse pairs counted once; self-loops excluded; BOTH endpoints must be labelled.
      Relation  Total labelled edges  Different-label edges  Same-label edges Global heterophily
         U-P-U                147382                  31655            115727           0.214782
         U-S-U               2799549                 124232           2675317           0.044376
         U-V-U                693044                  26970            666074           0.038915
Combined union               3299291                 168920           3130371           0.051199


In [6]:
# CELL 6 — LOCAL HETEROPHILY ON LABELLED SUBGRAPH
def local_h_summary(adj, name):
    A = adj.tocsr(copy=False)
    vals=[]; fraud_vals=[]; normal_vals=[]
    for i in range(3305, num_nodes):
        nbr=A.indices[A.indptr[i]:A.indptr[i+1]]
        nbr=nbr[(nbr>=3305) & (nbr!=i)]
        if len(nbr)==0: continue
        h=float(np.mean(labels[nbr] != labels[i])); vals.append(h)
        (fraud_vals if labels[i]==1 else normal_vals).append(h)
    x=np.asarray(vals)
    return {"Relation":name,"Labelled nodes with degree > 0":len(x),
            "Isolated labelled nodes":labelled_nodes-len(x),"Mean local H":x.mean(),
            "Median local H":np.median(x),"Std local H":x.std(ddof=0),
            "Q1 local H":np.percentile(x,25),"Q3 local H":np.percentile(x,75),
            "Min local H":x.min(),"Max local H":x.max(),
            "Fraud-node mean H":np.mean(fraud_vals),"Normal-node mean H":np.mean(normal_vals)}

local_results=pd.DataFrame([local_h_summary(net_upu,"U-P-U"),local_h_summary(net_usu,"U-S-U"),
                            local_h_summary(net_uvu,"U-V-U"),local_h_summary(union,"Combined union")])
print("===== LOCAL HETEROPHILY =====")
print("Nodes with no labelled neighbours are excluded; population SD (ddof=0).")
print(local_results.to_string(index=False, formatters={c:(lambda x:f"{x:.6f}") for c in local_results.columns if " H" in c}))

===== LOCAL HETEROPHILY =====
Nodes with no labelled neighbours are excluded; population SD (ddof=0).
      Relation  Labelled nodes with degree > 0  Isolated labelled nodes Mean local H Median local H Std local H Q1 local H Q3 local H Min local H Max local H Fraud-node mean H Normal-node mean H
         U-P-U                            7838                      801     0.128707       0.000000    0.263806   0.000000   0.125000    0.000000    1.000000          0.886546           0.056249
         U-S-U                            8549                       90     0.111765       0.023109    0.247451   0.013127   0.050157    0.000000    1.000000          0.841862           0.035141
         U-V-U                            8562                       77     0.102391       0.017544    0.243101   0.000000   0.032258    0.000000    1.000000          0.790798           0.031831
Combined union                            8639                        0     0.114242       0.024141    0.248737   0.01

In [7]:
# CELL 7 — ORIGINAL/REFERENCE CARE-GNN SPLIT
from sklearn.model_selection import train_test_split
index=np.arange(3305,num_nodes)
idx_train, idx_test, y_train, y_test=train_test_split(index, labels[3305:], stratify=labels[3305:],
                                                      test_size=0.60, random_state=2, shuffle=True)
print("===== SPLIT =====")
print("Raw Amazon.mat fixed split: None")
print("CARE-GNN eligible nodes: IDs 3305–11943 (first 3305 are excluded as unlabeled)")
print("Train: 40% =",len(idx_train))
print("Validation: None / 0%")
print("Test: 60% =",len(idx_test))
print("Split type: Random, stratified; random_state=2; NOT temporal")
print(f"Train fraud %: {(labels[idx_train]==1).mean()*100:.4f}%")
print(f"Test fraud %: {(labels[idx_test]==1).mean()*100:.4f}%")

===== SPLIT =====
Raw Amazon.mat fixed split: None
CARE-GNN eligible nodes: IDs 3305–11943 (first 3305 are excluded as unlabeled)
Train: 40% = 3455
Validation: None / 0%
Test: 60% = 5184
Split type: Random, stratified; random_state=2; NOT temporal
Train fraud %: 9.4935%
Test fraud %: 9.5100%


## Amazon-specific problems found during investigation

### Problem 1 — The first 3,305 nodes are not verified normal nodes
`Amazon.mat` stores 0/1 values in its label vector for all 11,944 graph nodes. However, the original CARE-GNN `train.py` explicitly says **0–3304 are unlabeled nodes** and performs supervised splitting only on IDs **3305–11943**.

Therefore this notebook does **not** count the first 3,305 stored zeros as confirmed normal users.

**Correct labelled benchmark counts used here:**
- Labelled nodes: **8,639**
- Fraud nodes: **821**
- Normal nodes: **7,818**
- Fraud rate: **9.5034%**
- Normal:Fraud imbalance: **9.5225:1**

### Problem 2 — Heterophily can be distorted by the unlabeled nodes
If the first 3,305 stored zeros are treated as real normal labels, same/different-label edge calculations can be wrong.

For **global heterophily**, this notebook therefore includes an edge only when **both endpoints are CARE-GNN-labelled nodes**.

For **local heterophily**, a labelled node is compared only with its **labelled neighbours**.

### Problem 3 — Edge count depends on the counting convention
The three sparse relation matrices contain stored adjacency entries, reverse/bidirectional entries, and overlapping structural edges across relations.

To make Amazon comparable with the YelpChi report, the master-table edge count is the **deduplicated unique undirected union of U-P-U, U-S-U and U-V-U**, with self-loops excluded.

Raw relation-wise stored counts are kept separately.

### Problem 4 — A time condition does not make this a dynamic graph
U-S-U uses a **one-week condition** when forming a relation, but the released benchmark is still one static graph snapshot. It is therefore recorded as **Static**, not dynamic/temporal.

### Problem 5 — No intrinsic fixed train/validation/test masks
The raw `Amazon.mat` file does not provide one universal benchmark split. For the master table, this notebook follows the **original CARE-GNN implementation**:

- supervised nodes: IDs 3305–11943 only
- train: **40%**
- validation: **0% / none**
- test: **60%**
- split: **random, stratified**
- `random_state=2`
- temporal split: **No**

### Problem 6 — Repository corrections matter later
The CARE-GNN repository documents corrections to some reported similarity scores and a CARE-Weight relation-weight conclusion. Those corrections do **not** change the dataset statistics calculated in this notebook, but they should be remembered before doing the later model-performance comparison.


In [8]:
# CELL 8 — SOURCES, LINKS, LIMITATIONS
metadata={
"Paper":"https://arxiv.org/abs/2008.08692",
"Official dataset download":"https://data.dgl.ai/dataset/FraudAmazon.zip",
"CARE-GNN GitHub":"https://github.com/YingtongDou/CARE-GNN",
"Important limitations":"First 3,305 graph nodes are treated as unlabeled by CARE-GNN even though the MAT label array stores zeros there. Fraud % should therefore be reported on the 8,639 labelled nodes. Edge totals depend on whether relation duplicates/reverse edges are counted. Labels are real-world proxy fraud/anomaly labels rather than synthetic injections. Raw MAT file has no universal fixed train/validation/test masks.",
"Compatibility":"Designed for same-node-type, multi-relational fraud GNNs (Amazon/YelpChi style). Some libraries may require conversion from scipy sparse MAT matrices to their graph format."}
for k,v in metadata.items(): print(k+":",v)

Paper: https://arxiv.org/abs/2008.08692
Official dataset download: https://data.dgl.ai/dataset/FraudAmazon.zip
CARE-GNN GitHub: https://github.com/YingtongDou/CARE-GNN
Important limitations: First 3,305 graph nodes are treated as unlabeled by CARE-GNN even though the MAT label array stores zeros there. Fraud % should therefore be reported on the 8,639 labelled nodes. Edge totals depend on whether relation duplicates/reverse edges are counted. Labels are real-world proxy fraud/anomaly labels rather than synthetic injections. Raw MAT file has no universal fixed train/validation/test masks.
Compatibility: Designed for same-node-type, multi-relational fraud GNNs (Amazon/YelpChi style). Some libraries may require conversion from scipy sparse MAT matrices to their graph format.


In [9]:
# CELL 9 — FINAL COMMON TABLE ROW
g=global_results[global_results.Relation=="Combined union"].iloc[0]
l=local_results[local_results.Relation=="Combined union"].iloc[0]
amazon_final=pd.DataFrame([{
"Dataset":"Amazon", "Domain":"Amazon review/user fraud detection",
"Nodes":num_nodes,"Edges":unique_undirected_edges,"Features":num_features,
"Fraud / anomaly nodes":fraud_count,"Normal nodes":normal_count,"Unlabelled nodes":unlabelled_count,
"Fraud %":fraud_pct,"Imbalance ratio (Normal:Fraud)":f"{imbalance_ratio:.4f}:1",
"Graph type":"Homogeneous node type, multi-relational","Node types":1,"Node type":"User","Relation types":3,
"Relations":"U-P-U: same product; U-S-U: same star rating within one week; U-V-U: top-5% mutual review-text similarity",
"Static / Dynamic":"Static","Anomaly origin":"Real-world / non-injected proxy fraud labels",
"Global heterophily":float(g["Global heterophily"]),
"Local heterophily mean":float(l["Mean local H"]),"Local heterophily median":float(l["Median local H"]),
"Local heterophily std":float(l["Std local H"]),"Fraud-node mean local H":float(l["Fraud-node mean H"]),
"Normal-node mean local H":float(l["Normal-node mean H"]),"Isolated labelled nodes":int(l["Isolated labelled nodes"]),
"Raw fixed split":"None","Reference implementation":"CARE-GNN","Train split":"40%","Validation split":"None / 0%",
"Test split":"60%","Split type":"Random, stratified","Split seed / random_state":2,"Temporal split":"No",
"Dataset / paper link":"https://arxiv.org/abs/2008.08692",
"Download link":"https://data.dgl.ai/dataset/FraudAmazon.zip","GitHub":"https://github.com/YingtongDou/CARE-GNN",
"Important limitation / compatibility":metadata["Important limitations"]+" "+metadata["Compatibility"]
}])
pd.set_option("display.max_columns",None); pd.set_option("display.max_colwidth",None)
display(amazon_final)
amazon_final.to_csv("/kaggle/working/amazon_final_dataset_row.csv",index=False)
print("Saved /kaggle/working/amazon_final_dataset_row.csv")

,Dataset,Domain,Nodes,Edges,Features,Fraud / anomaly nodes,Normal nodes,Unlabelled nodes,Fraud %,Imbalance ratio (Normal:Fraud),Graph type,Node types,Node type,Relation types,Relations,Static / Dynamic,Anomaly origin,Global heterophily,Local heterophily mean,Local heterophily median,Local heterophily std,Fraud-node mean local H,Normal-node mean local H,Isolated labelled nodes,Raw fixed split,Reference implementation,Train split,Validation split,Test split,Split type,Split seed / random_state,Temporal split,Dataset / paper link,Download link,GitHub,Important limitation / compatibility
0,Amazon,Amazon review/user fraud detection,11944,4417576,25,821,7818,3305,9.503415,9.5225:1,"Homogeneous node type, multi-relational",1,User,3,U-P-U: same product; U-S-U: same star rating within one week; U-V-U: top-5% mutual review-text similarity,Static,Real-world / non-injected proxy fraud labels,0.051199,0.114242,0.024141,0.248737,0.862603,0.035654,0,None,CARE-GNN,40%,None / 0%,60%,"Random, stratified",2,No,https://arxiv.org/abs/2008.08692,https://data.dgl.ai/dataset/FraudAmazon.zip,https://github.com/YingtongDou/CARE-GNN,"First 3,305 graph nodes are treated as unlabeled by CARE-GNN even though the MAT label array stores zeros there. Fraud % should therefore be reported on the 8,639 labelled nodes. Edge totals depend on whether relation duplicates/reverse edges are counted. Labels are real-world proxy fraud/anomaly labels rather than synthetic injections. Raw MAT file has no universal fixed train/validation/test masks. Designed for same-node-type, multi-relational fraud GNNs (Amazon/YelpChi style). Some libraries may require conversion from scipy sparse MAT matrices to their graph format."


Saved /mnt/data/amazon_final_dataset_row.csv


## Final interpretation

This notebook is the **dataset-investigation first deliverable**, not a model benchmark. The master-table edge count uses the **unique undirected union graph** across the three relations. Heterophily is calculated only where **both endpoints are labelled**, because CARE-GNN explicitly excludes node IDs 0–3304 from the Amazon labelled set. This avoids treating the placeholder zeros for those nodes as verified normal labels.

**Global heterophily formula:** different-label labelled edges ÷ total labelled edges.

For local heterophily, each labelled node’s value is the fraction of its labelled neighbours having the opposite label; mean, median and population standard deviation are then reported over labelled nodes with at least one labelled neighbour.

## Limitations, Key Findings and Dataset Limitations / Compatibility

### Key Findings
- The Amazon graph contains **11,944 user nodes** and **25 node features**.
- The original CARE-GNN protocol treats node IDs **0–3304 as unlabeled**, leaving **8,639 labelled nodes** for supervised evaluation.
- Within the labelled subset there are **821 fraud/anomaly nodes** and **7,818 normal nodes**, so the dataset is strongly class-imbalanced.
- The graph has **one node type (User)** and **three relation/edge types: U-P-U, U-S-U and U-V-U**.
- It is a **static, homogeneous-node-type, multi-relational graph**.
- Fraud/anomaly labels are treated as **real-world / non-injected**, rather than synthetically injected anomalies.
- Global heterophily is calculated as **different-label edges ÷ total labelled edges**.
- Local heterophily is calculated per labelled node using its labelled neighbours, with **mean, median and population standard deviation** reported.
- The reference CARE-GNN split is **40% train, no validation split, and 60% test**, using a random stratified split with `random_state=2`.

### Limitations
- **Unlabeled-node issue:** the first 3,305 nodes contain stored zero values in the MAT label vector, but CARE-GNN explicitly treats them as unlabeled. Counting them as confirmed normal nodes would distort fraud percentage, imbalance and heterophily.
- **No intrinsic fixed split:** `Amazon.mat` itself does not contain one universal train/validation/test mask. The reported split therefore follows the original CARE-GNN implementation.
- **Edge-count ambiguity:** relation-wise sparse entries, reverse/bidirectional entries and overlapping edges can produce different edge totals depending on the counting convention. This notebook uses the deduplicated unique undirected union for the common table.
- **Static graph:** although the U-S-U relation uses a one-week condition during graph construction, the released benchmark is a static snapshot rather than a dynamic temporal graph.
- **Benchmark/repository caveat:** the CARE-GNN repository documents corrections to some reported similarity scores and a CARE-Weight conclusion. This does not change the dataset statistics here, but it matters for later model-performance analysis.

### Dataset Limitations / Compatibility
- The dataset is particularly suitable for **fraud-specific GNN research involving homogeneous user nodes with multiple relation types**.
- Models expecting a single homogeneous adjacency matrix may require the three Amazon relations to be merged or otherwise transformed.
- Models designed for heterogeneous graphs with multiple **node types** cannot use the dataset's structure directly without adaptation, because Amazon has only one node type.
- Modern **PyTorch Geometric or DGL** pipelines may require conversion from the provided MATLAB/scipy sparse representation.
- The CARE-GNN unlabeled-node convention must be preserved when comparing models; otherwise different preprocessing choices can make results incomparable.
- The dataset does not provide a genuine temporal evaluation setting, so it is not directly suitable for testing temporal/dynamic fraud-GNN methods without additional temporal data.
